# Strategy Engines: Overview & Comparison

quaver ships with **7 single-asset** and **1 multi-asset** strategy engines,
all registered in `StrategyRegistry` and ready to use with `run_backtest()`.

This notebook:
1. Lists all registered engines and their parameters
2. Runs each single-asset strategy on the same data
3. Compares results side-by-side
4. Demonstrates the pairs trading strategy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from quaver.strategies.registry import StrategyRegistry
import quaver.strategies  # noqa: F401 -- triggers auto-registration
from quaver.backtest import run_backtest, run_multi_asset_backtest
from quaver.backtest.portfolio import CommissionConfig, ExitRules

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Strategy Registry

In [ ]:
engines = StrategyRegistry.list_engines()
print(f"Registered engines ({len(engines)}):")
print()
for name in engines:
    cls = StrategyRegistry.get(name)
    kind = StrategyRegistry.get_strategy_kind(name)
    defaults = cls.get_default_parameters()
    print(f"  {name} [{kind}]")
    print(f"    {cls.display_name}: {cls.description[:80]}...")
    print(f"    Defaults: {defaults}")
    print()

## 2. Generate Synthetic Market Data

We generate 600 bars with trend, mean-reversion, and volatility regimes
so that different strategy types can each find opportunities.

In [ ]:
def generate_market_data(n=600, seed=42):
    """Generate synthetic OHLCV with trend+oscillation for strategy testing."""
    np.random.seed(seed)
    # Combine uptrend + oscillation + noise
    trend = np.linspace(0, 30, n)
    oscillation = 15 * np.sin(np.linspace(0, 10 * np.pi, n))
    noise = np.cumsum(np.random.randn(n) * 0.5)
    close = 100 + trend + oscillation + noise
    close = np.maximum(close, 10)  # floor at 10

    # Realistic OHLV
    high = close + np.abs(np.random.randn(n) * 2.0)
    low = close - np.abs(np.random.randn(n) * 2.0)
    open_ = close + np.random.randn(n) * 0.5
    vol_base = 1_000_000
    volume = np.abs(vol_base + np.random.randn(n) * 400_000 +
                    200_000 * np.sin(np.linspace(0, 4 * np.pi, n)))

    rows = []
    for i in range(n):
        rows.append({
            "ts": datetime(2023, 1, 1) + timedelta(days=i),
            "open": open_[i],
            "high": max(high[i], open_[i], close[i]),
            "low": min(low[i], open_[i], close[i]),
            "close": close[i],
            "volume": volume[i],
        })
    return pd.DataFrame(rows)


candles = generate_market_data()
print(f"Generated {len(candles)} bars")
print(f"Price range: {candles['close'].min():.1f} - {candles['close'].max():.1f}")

fig, ax = plt.subplots()
ax.plot(candles["ts"], candles["close"], linewidth=0.6)
ax.set_title("Synthetic Market Data")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
plt.tight_layout()
plt.show()

## 3. Run All Single-Asset Strategies

In [ ]:
single_engines = [
    name for name in engines
    if StrategyRegistry.get_strategy_kind(name) == "single"
]

results = {}
for name in single_engines:
    cls = StrategyRegistry.get(name)
    params = cls.get_default_parameters()
    try:
        result = run_backtest(
            engine_name=name,
            parameters=params,
            candles=candles,
            instrument_id="SYN",
            initial_capital=10_000,
            quantity_per_trade=10,
            commission=CommissionConfig(pct_of_notional=0.001),
            exit_rules=ExitRules(stop_loss_pct=0.05),
        )
        results[name] = result
        print(f"{name:30s} trades={result.total_trades:3d}  "
              f"return={result.total_return*100:+7.2f}%  "
              f"win_rate={result.win_rate*100:5.1f}%")
    except Exception as e:
        print(f"{name:30s} ERROR: {e}")

## 4. Strategy Comparison Table

In [ ]:
if results:
    comparison = pd.DataFrame({name: r.summary() for name, r in results.items()}).T
    display_cols = [
        "total_trades", "total_return_pct", "win_rate_pct",
        "avg_pnl", "sharpe_ratio", "profit_factor",
        "max_drawdown_pct", "expectancy",
    ]
    comparison[display_cols]

## 5. Equity Curves Side-by-Side

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(14, 6))
    for name, result in results.items():
        cpnl = result.cumulative_pnl
        if cpnl:
            ax.plot(range(len(cpnl)), cpnl, linewidth=1.5, label=f"{name} ({result.total_trades} trades)")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Trade #")
    ax.set_ylabel("Cumulative P&L ($)")
    ax.set_title("Strategy Comparison: Equity Curves")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. Pairs Trading (Multi-Asset Strategy)

The `pairs_mean_reversion` strategy trades the spread between two instruments.
It uses z-score entry/exit thresholds on the rolling normalised spread.

In [ ]:
# Generate two correlated price series
candles_a = generate_market_data(n=600, seed=42)
candles_b = generate_market_data(n=600, seed=43)
# Make B correlated with A but with some divergence
candles_b["close"] = candles_a["close"] * 0.9 + candles_b["close"] * 0.1 + 10
candles_b["high"] = candles_b["close"] + np.abs(np.random.randn(600) * 2)
candles_b["low"] = candles_b["close"] - np.abs(np.random.randn(600) * 2)
candles_b["open"] = candles_b["close"] + np.random.randn(600) * 0.3

fig, ax = plt.subplots()
ax.plot(candles_a["ts"], candles_a["close"], label="Instrument A", linewidth=0.8)
ax.plot(candles_b["ts"], candles_b["close"], label="Instrument B", linewidth=0.8)
ax.set_title("Two Correlated Instruments")
ax.legend()
ax.set_xlabel("Date")
ax.set_ylabel("Price")
plt.tight_layout()
plt.show()

In [ ]:
pairs_results = run_multi_asset_backtest(
    engine_name="pairs_mean_reversion",
    parameters={
        "instrument_a": "A",
        "instrument_b": "B",
        "spread_window": 60,
        "entry_z": 2.0,
        "exit_z": 0.5,
    },
    candles_map={"A": candles_a, "B": candles_b},
    initial_capital=10_000,
    quantity_per_trade=10,
    allow_shorting=True,
)

for iid, r in pairs_results.items():
    s = r.summary()
    print(f"\n{iid}: {s['total_trades']} trades, "
          f"return={s['total_return_pct']:+.2f}%, "
          f"win_rate={s['win_rate_pct']:.1f}%")

In [ ]:
# Visualise spread z-score
merged = pd.merge(
    candles_a[["ts", "close"]].rename(columns={"close": "a"}),
    candles_b[["ts", "close"]].rename(columns={"close": "b"}),
    on="ts",
)
spread = merged["a"] - merged["b"]
roll_mean = spread.rolling(60).mean()
roll_std = spread.rolling(60).std(ddof=1)
z = (spread - roll_mean) / roll_std

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(merged["ts"], spread, linewidth=0.8)
axes[0].plot(merged["ts"], roll_mean, linestyle="--", linewidth=0.8)
axes[0].set_title("Price Spread (A - B)")
axes[0].set_ylabel("Spread ($)")

axes[1].plot(merged["ts"], z, linewidth=0.8, color="tab:orange")
axes[1].axhline(2.0, color="red", linestyle="--", linewidth=0.8, label="Entry z=+/-2")
axes[1].axhline(-2.0, color="red", linestyle="--", linewidth=0.8)
axes[1].axhline(0.5, color="green", linestyle=":", linewidth=0.8, label="Exit z=+/-0.5")
axes[1].axhline(-0.5, color="green", linestyle=":", linewidth=0.8)
axes[1].axhline(0, color="grey", linewidth=0.5)
axes[1].set_title("Spread Z-Score")
axes[1].set_ylabel("Z-Score")
axes[1].set_xlabel("Date")
axes[1].legend()
plt.tight_layout()
plt.show()

## Strategy Summary

| Engine | Type | Logic |
|--------|------|-------|
| `mean_reversion` | Single | Dual MA crossover with threshold |
| `regime_mean_reversion` | Single | ADX/BB regime classification + conditional probability |
| `vsa_stopping_volume` | Single | Volume Spread Analysis pattern detection |
| `breakout_consolidation` | Single | Consolidation range breakout with volume confirmation |
| `pullback_trend` | Single | Trend continuation at MA pullback with RSI dip |
| `reversal_support` | Single | Counter-trend mean reversion at support levels |
| `pairs_mean_reversion` | Multi | Z-score spread trading between two instruments |